# Entraînement du World Model — Bras de découpe

Notebook Colab pour générer le dataset et entraîner le modèle de monde du bras de découpe.  
**Runtime recommandé : GPU (T4 ou A100)**

Pipeline :
```
1. Clone du repo GitHub
2. Installation des dépendances
3. Génération des pièces  →  pieces_database.json
4. Simulation & enregistrement  →  dataset/episode_*.npz
5. Entraînement du world model  →  checkpoints/best_model.pt
6. Visualisation des courbes
```

## 0. Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Aucun GPU détecté. Aller dans Exécution → Modifier le type d'exécution → GPU.")

## 1. Montage Google Drive (optionnel mais recommandé)

Monte Drive pour sauvegarder dataset et checkpoints entre sessions.  
**Passer cette cellule si tu ne veux pas utiliser Drive.**

In [ ]:
USE_DRIVE = True  # Mettre False pour ne pas utiliser Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/chain_simple_decoupe'
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Drive monté → {DRIVE_DIR}")
else:
    DRIVE_DIR = None
    print("Drive non utilisé — les fichiers seront perdus à la fin de la session.")

## 2. Clone du repo et installation

In [ ]:
import os

REPO_URL  = 'https://github.com/JulesV19/industrial-world-model.git'
REPO_DIR  = '/content/industrial-world-model'
WORK_DIR  = os.path.join(REPO_DIR, 'arm_decoupe')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(WORK_DIR)
print(f"Répertoire de travail : {os.getcwd()}")

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
# pygame n'est pas utile en Colab (pas d'affichage), on l'ignore silencieusement
print("Dépendances installées.")

## 3. Paramètres du run

Ajuste ces valeurs selon le temps et la VRAM disponibles.

In [ ]:
# --- Génération du dataset ---
NUM_PIECES   = 100   # Nombre de pièces uniques
N_RUNS       = 5     # Runs de simulation par pièce  →  NUM_PIECES × N_RUNS épisodes
SPEED_MIN    = 1.0   # Durée min par segment (s)
SPEED_MAX    = 3.0   # Durée max par segment (s)
SEED         = 42

# --- Entraînement ---
EPOCHS          = 100
BATCH_SIZE      = 32
LR              = 3e-4
EARLY_STOPPING  = 40    # Patience (epochs sans amélioration)
VAL_SPLIT       = 0.1

# Archi
SHAPE_EMBED_DIM = 256
H_DIM           = 512
GRU_LAYERS      = 3

# Chemins
DATASET_DIR   = 'dataset'
DB_PATH       = 'pieces_database.json'
SAVE_DIR      = 'world_model/checkpoints'

print(f"Dataset : {NUM_PIECES} pièces × {N_RUNS} runs = {NUM_PIECES * N_RUNS} épisodes")

## 4. Génération des pièces

Produit `pieces_database.json` avec les waypoints de découpe.

In [ ]:
import os

# Récupération depuis Drive si disponible
if USE_DRIVE and os.path.exists(f'{DRIVE_DIR}/{DB_PATH}'):
    import shutil
    shutil.copy(f'{DRIVE_DIR}/{DB_PATH}', DB_PATH)
    print(f"pieces_database.json restauré depuis Drive.")
else:
    !python generate_pieces.py \
        -n {NUM_PIECES} \
        --n-runs {N_RUNS} \
        --speed-min {SPEED_MIN} \
        --speed-max {SPEED_MAX} \
        --seed {SEED}

    if USE_DRIVE:
        import shutil
        shutil.copy(DB_PATH, f'{DRIVE_DIR}/{DB_PATH}')
        print(f"Sauvegardé sur Drive : {DRIVE_DIR}/{DB_PATH}")

## 5. Simulation et enregistrement du dataset

Simule le bras avec physique réaliste et enregistre les trajectoires.  
⚠️ Cette étape prend ~1–2 min pour 100 pièces × 5 runs.

In [ ]:
import os, glob, shutil

os.makedirs(DATASET_DIR, exist_ok=True)
existing_episodes = glob.glob(f'{DATASET_DIR}/episode_*.npz')
expected_episodes = NUM_PIECES * N_RUNS

# Restauration depuis Drive si le dataset est déjà complet
if USE_DRIVE and len(existing_episodes) < expected_episodes:
    drive_episodes = glob.glob(f'{DRIVE_DIR}/dataset/episode_*.npz')
    if len(drive_episodes) == expected_episodes:
        print(f"Dataset complet trouvé sur Drive ({len(drive_episodes)} épisodes) — copie en cours...")
        os.makedirs(DATASET_DIR, exist_ok=True)
        for f in drive_episodes:
            shutil.copy(f, DATASET_DIR)
        existing_episodes = glob.glob(f'{DATASET_DIR}/episode_*.npz')
        print("Dataset restauré.")

if len(existing_episodes) >= expected_episodes:
    print(f"Dataset déjà complet ({len(existing_episodes)} épisodes). Simulation ignorée.")
else:
    print(f"Génération de {expected_episodes} épisodes...")
    !python record_dataset.py

    if USE_DRIVE:
        drive_dataset_dir = f'{DRIVE_DIR}/dataset'
        os.makedirs(drive_dataset_dir, exist_ok=True)
        for f in glob.glob(f'{DATASET_DIR}/episode_*.npz'):
            shutil.copy(f, drive_dataset_dir)
        print(f"Dataset sauvegardé sur Drive : {drive_dataset_dir}")

final_count = len(glob.glob(f'{DATASET_DIR}/episode_*.npz'))
print(f"\nTotal épisodes disponibles : {final_count}")

## 6. Entraînement du World Model

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

!python world_model/train.py \
    --data_dir        {DATASET_DIR} \
    --db_path         {DB_PATH} \
    --save_dir        {SAVE_DIR} \
    --epochs          {EPOCHS} \
    --batch_size      {BATCH_SIZE} \
    --lr              {LR} \
    --early_stopping  {EARLY_STOPPING} \
    --val_split       {VAL_SPLIT} \
    --shape_embed_dim {SHAPE_EMBED_DIM} \
    --h_dim           {H_DIM} \
    --gru_layers      {GRU_LAYERS} \
    --seed            {SEED} \
    --use_amp True \
    --num_workers 2

## 7. Sauvegarde des checkpoints sur Drive

In [ ]:
if USE_DRIVE:
    import shutil
    drive_ckpt_dir = f'{DRIVE_DIR}/checkpoints'
    if os.path.exists(SAVE_DIR):
        shutil.copytree(SAVE_DIR, drive_ckpt_dir, dirs_exist_ok=True)
        print(f"Checkpoints sauvegardés sur Drive : {drive_ckpt_dir}")
        for f in os.listdir(drive_ckpt_dir):
            print(f"  {f}")
    else:
        print("Aucun checkpoint trouvé.")
else:
    print("Drive non utilisé — télécharge manuellement les fichiers dans", SAVE_DIR)

## 8. Visualisation des courbes d'entraînement

In [ ]:
from IPython.display import Image, display
import os

curves_path = os.path.join(SAVE_DIR, 'training_curves.png')
if os.path.exists(curves_path):
    display(Image(curves_path))
else:
    print("Courbes introuvables — l'entraînement a peut-être échoué.")

## 9. Résumé du modèle entraîné

In [ ]:
import torch, os

ckpt_path = os.path.join(SAVE_DIR, 'best_model.pt')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print("=== Modèle sauvegardé ===")
    if 'best_val_loss' in ckpt:
        print(f"  Meilleure val loss  : {ckpt['best_val_loss']:.6f}")
    if 'epoch' in ckpt:
        print(f"  Epoch               : {ckpt['epoch']}")
    if 'hparams' in ckpt:
        print("  Hyperparamètres :")
        for k, v in ckpt['hparams'].items():
            print(f"    {k}: {v}")
    n_params = sum(p.numel() for p in torch.load(ckpt_path, map_location='cpu')['model_state_dict'].values())
    print(f"  Paramètres totaux   : {n_params:,}")
else:
    print("Checkpoint introuvable.")